Gold Layer

In [0]:
%sql
   SELECT COUNT(*) AS row_count, COUNT(DISTINCT PARCEL_PK) AS distinct_pk
   FROM parcels_project.silver.property_cleaned;

row_count,distinct_pk
437403,437403


In [0]:
%sql
   CREATE SCHEMA IF NOT EXISTS parcels_project.gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_city (
  city_key INT,
  city_name STRING,
  PLANNING_JURISDICTION STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_city (city_key, city_name, PLANNING_JURISDICTION)
SELECT
  (SELECT COALESCE(MAX(city_key), 0) FROM parcels_project.gold.dim_city) + ROW_NUMBER() OVER (ORDER BY city_name),
  city_name,
  PLANNING_JURISDICTION
FROM (
  SELECT DISTINCT CITY_DECODE AS city_name, PLANNING_JURISDICTION
  FROM parcels_project.silver.property_cleaned
  WHERE CITY_DECODE IS NOT NULL
    AND CITY_DECODE NOT IN (SELECT city_name FROM parcels_project.gold.dim_city)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_township (
  township_key INT,
  township_name STRING,
  FIREDIST STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_township (township_key, township_name, FIREDIST)
SELECT
  (SELECT COALESCE(MAX(township_key), 0) FROM parcels_project.gold.dim_township) + ROW_NUMBER() OVER (ORDER BY township_name),
  township_name,
  FIREDIST
FROM (
  SELECT township_name, FIREDIST
  FROM (
    SELECT
      TOWNSHIP_DECODE AS township_name,
      FIREDIST,
      ROW_NUMBER() OVER (PARTITION BY TOWNSHIP_DECODE ORDER BY FIREDIST) AS rn
    FROM parcels_project.silver.property_cleaned
    WHERE TOWNSHIP_DECODE IS NOT NULL
  ) t
  WHERE rn = 1
    AND township_name NOT IN (SELECT township_name FROM parcels_project.gold.dim_township)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_land_class (
  land_class_key INT,
  land_class_code STRING,
  land_class_name STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_land_class (land_class_key, land_class_code, land_class_name)
SELECT
  (SELECT COALESCE(MAX(land_class_key), 0) FROM parcels_project.gold.dim_land_class) + ROW_NUMBER() OVER (ORDER BY land_class_name),
  land_class_code,
  land_class_name
FROM (
  SELECT land_class_code, land_class_name
  FROM (
    SELECT
      LAND_CLASS AS land_class_code,
      LAND_CLASS_DECODE AS land_class_name,
      ROW_NUMBER() OVER (PARTITION BY LAND_CLASS_DECODE ORDER BY LAND_CLASS) AS rn
    FROM parcels_project.silver.property_cleaned
    WHERE LAND_CLASS_DECODE IS NOT NULL
  ) t
  WHERE rn = 1
    AND land_class_name NOT IN (SELECT land_class_name FROM parcels_project.gold.dim_land_class)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_property_type (
  property_type_key INT,
  type_use_code STRING,
  type_use_name STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_property_type (property_type_key, type_use_code, type_use_name)
SELECT
  (SELECT COALESCE(MAX(property_type_key), 0) FROM parcels_project.gold.dim_property_type) + ROW_NUMBER() OVER (ORDER BY type_use_name),
  type_use_code,
  type_use_name
FROM (
  SELECT type_use_code, type_use_name
  FROM (
    SELECT
      TYPE_AND_USE AS type_use_code,
      TYPE_USE_DECODE AS type_use_name,
      ROW_NUMBER() OVER (PARTITION BY TYPE_USE_DECODE ORDER BY TYPE_AND_USE) AS rn
    FROM parcels_project.silver.property_cleaned
    WHERE TYPE_USE_DECODE IS NOT NULL
  ) t
  WHERE rn = 1
    AND type_use_name NOT IN (SELECT type_use_name FROM parcels_project.gold.dim_property_type)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_design_style (
  design_style_key INT,
  design_style_code STRING,
  design_style_name STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_design_style (design_style_key, design_style_code, design_style_name)
SELECT
  (SELECT COALESCE(MAX(design_style_key), 0) FROM parcels_project.gold.dim_design_style) + ROW_NUMBER() OVER (ORDER BY design_style_name),
  design_style_code,
  design_style_name
FROM (
  SELECT design_style_code, design_style_name
  FROM (
    SELECT
      DESIGNSTYL AS design_style_code,
      DESIGN_STYLE_DECODE AS design_style_name,
      ROW_NUMBER() OVER (PARTITION BY DESIGN_STYLE_DECODE ORDER BY DESIGNSTYL) AS rn
    FROM parcels_project.silver.property_cleaned
    WHERE DESIGN_STYLE_DECODE IS NOT NULL
  ) t
  WHERE rn = 1
    AND design_style_name NOT IN (SELECT design_style_name FROM parcels_project.gold.dim_design_style)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.dim_billing_class (
  billing_class_key INT,
  billing_class_code STRING,
  billing_class_name STRING
) USING DELTA

In [0]:
%sql
INSERT INTO parcels_project.gold.dim_billing_class (billing_class_key, billing_class_code, billing_class_name)
SELECT
  (SELECT COALESCE(MAX(billing_class_key), 0) FROM parcels_project.gold.dim_billing_class) + ROW_NUMBER() OVER (ORDER BY billing_class_name),
  billing_class_code,
  billing_class_name
FROM (
  SELECT billing_class_code, billing_class_name
  FROM (
    SELECT
      BILLCLASS AS billing_class_code,
      BILLING_CLASS_DECODE AS billing_class_name,
      ROW_NUMBER() OVER (PARTITION BY BILLING_CLASS_DECODE ORDER BY BILLCLASS) AS rn
    FROM parcels_project.silver.property_cleaned
    WHERE BILLING_CLASS_DECODE IS NOT NULL
  ) t
  WHERE rn = 1
    AND billing_class_name NOT IN (SELECT billing_class_name FROM parcels_project.gold.dim_billing_class)
)

num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS parcels_project.gold.fact_property (
    PARCEL_PK STRING,
    city_key INT,
    township_key INT,
    land_class_key INT,
    property_type_key INT,
    design_style_key INT,
    billing_class_key INT,
    BUILD_ERA STRING,
    IS_TAX_EXEMPT BOOLEAN,
    SITE_ADDRESS STRING,
    STREET_ADDRESS_FULL STRING,
    OWNER STRING,
    YEAR_BUILT INT,
    PROPERTY_AGE_YEARS INT,
    HEATEDAREA DOUBLE,
    DEED_ACRES DOUBLE,
    BLDG_VAL DOUBLE,
    LAND_VAL DOUBLE,
    TOTAL_VALUE_ASSD DOUBLE,
    ASSESSED_VALUE_PER_SQFT DOUBLE,
    ASSESSED_VALUE_PER_ACRE DOUBLE,
    TOTSALPRICE DOUBLE,
    SALE_DATE DATE,
    DEED_DATE DATE,
    UNITS INT,
    TOTSTRUCTS INT,
    TOTUNITS INT
) USING DELTA

In [0]:
%sql
MERGE INTO parcels_project.gold.fact_property AS tgt
USING (
  SELECT
    p.PARCEL_PK, c.city_key, t.township_key, lc.land_class_key, pt.property_type_key,
    ds.design_style_key, bc.billing_class_key, p.BUILD_ERA, p.IS_TAX_EXEMPT,
    p.SITE_ADDRESS, p.STREET_ADDRESS_FULL, p.OWNER, p.YEAR_BUILT, p.PROPERTY_AGE_YEARS,
    p.HEATEDAREA, p.DEED_ACRES, p.BLDG_VAL, p.LAND_VAL, p.TOTAL_VALUE_ASSD,
    p.ASSESSED_VALUE_PER_SQFT, p.ASSESSED_VALUE_PER_ACRE, p.TOTSALPRICE,
    p.SALE_DATE, p.DEED_DATE, p.UNITS, p.TOTSTRUCTS, p.TOTUNITS
  FROM parcels_project.silver.property_cleaned p
  LEFT JOIN parcels_project.gold.dim_city          c  ON p.CITY_DECODE          <=> c.city_name
  LEFT JOIN parcels_project.gold.dim_township      t  ON p.TOWNSHIP_DECODE      <=> t.township_name
  LEFT JOIN parcels_project.gold.dim_land_class    lc ON p.LAND_CLASS_DECODE    <=> lc.land_class_name
  LEFT JOIN parcels_project.gold.dim_property_type pt ON p.TYPE_USE_DECODE      <=> pt.type_use_name
  LEFT JOIN parcels_project.gold.dim_design_style  ds ON p.DESIGN_STYLE_DECODE  <=> ds.design_style_name
  LEFT JOIN parcels_project.gold.dim_billing_class bc ON p.BILLING_CLASS_DECODE <=> bc.billing_class_name
) AS src
ON tgt.PARCEL_PK = src.PARCEL_PK
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
437403,437403,0,0


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM parcels_project.silver.property_cleaned) AS silver_rows,
  (SELECT COUNT(*) FROM parcels_project.gold.fact_property) AS fact_rows,
  (SELECT COUNT(DISTINCT PARCEL_PK) FROM parcels_project.gold.fact_property) AS fact_distinct_pk;

silver_rows,fact_rows,fact_distinct_pk
437403,437403,437403


In [0]:
%sql
SELECT COUNT(*) FROM parcels_project.gold.fact_property WHERE city_key IS NULL;

COUNT(*)
0


In [0]:
%sql

SELECT COUNT(*) 
FROM parcels_project.gold.fact_property 
WHERE city_key IS NULL;

COUNT(*)
0


In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM parcels_project.gold.fact_property f
JOIN parcels_project.silver.property_cleaned s
  ON f.PARCEL_PK = s.PARCEL_PK
WHERE s.source_file LIKE '%Incremantal_loading_1%';

row_count
0


In [0]:
%sql
SELECT * FROM parcels_project.gold.dim_city ORDER BY city_key DESC LIMIT 5

city_key,city_name,PLANNING_JURISDICTION
16,ZEBULON,WC
15,WENDELL,null
14,WAKE FOREST,WF
13,UNKNOWN,null
12,ROLESVILLE,RO
